# YOLOX Plugin live smoke test

This notebook sends real HTTP requests to the running YOLOX plugin and validates the health, metadata and prediction endpoints.

The service is expected to respond at http://127.0.0.1:8006.

In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import requests

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'plugin_framework').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PLUGIN_URL = os.environ.get('PLUGIN_URL', 'http://127.0.0.1:8006')
print(f'Using plugin URL: {PLUGIN_URL}')

Using plugin URL: http://127.0.0.1:8006


In [2]:
image = np.zeros((256, 256, 3), dtype=np.uint8)
image[:, :, 0] = 200
image[:, :, 1] = 120
image[:, :, 2] = 50
image[64:192, 64:192, :] = 255
print('image_shape=', image.shape)

image_shape= (256, 256, 3)


In [3]:
health = requests.get(f'{PLUGIN_URL}/health', timeout=15)
print('status_code=', health.status_code)
print(health.json())
assert health.status_code == 200
assert health.json().get('status') == 'ok'

status_code= 200
{'status': 'ok', 'name': 'yolox', 'version': '0.1.0', 'gpu': True}


In [4]:
metadata = requests.get(f'{PLUGIN_URL}/metadata', timeout=15)
print('status_code=', metadata.status_code)
meta = metadata.json()
print(meta.get('name'))
print(meta.get('version'))
assert metadata.status_code == 200
assert meta.get('name') == 'yolox'

status_code= 200
yolox
0.1.0


In [5]:
payload = {'image': image.tolist(), 'input_format': 'RGB'}
response = requests.post(f'{PLUGIN_URL}/predict', json=payload, timeout=120)
print('status_code=', response.status_code)
body = response.json()
print(json.dumps(body, indent=2)[:2000])
assert response.status_code == 200
assert 'num_detections' in body
assert 'detections' in body

status_code= 200
{
  "num_detections": 1,
  "detections": [
    {
      "bbox_xyxy": [
        61.007991790771484,
        58.90858459472656,
        193.21713256835938,
        196.24658203125
      ],
      "score": 0.9900316596031189,
      "class_id": 0,
      "class_name": "person"
    }
  ],
  "image_shape": [
    256,
    256,
    3
  ]
}


In [6]:
from PIL import Image
from pathlib import Path

png_path = Path('tmp_yolox_test.png')
Image.fromarray(image).save(png_path)
with png_path.open('rb') as fh:
    files = {'file': (png_path.name, fh, 'image/png')}
    infer_response = requests.post(f'{PLUGIN_URL}/infer', files=files, timeout=120)
    print('infer_status=', infer_response.status_code)
    print(infer_response.text[:1000])
    assert infer_response.status_code == 200

png_path.unlink(missing_ok=True)

infer_status= 200
{"num_detections":1,"detections":[{"bbox_xyxy":[60.838993072509766,62.798011779785156,192.3800506591797,195.78659057617188],"score":0.9798629283905029,"class_id":0,"class_name":"person"}],"image_shape":[256,256,3]}
